In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("raghavendragandhi/retail-customer-and-transaction-dataset")
print("Path to dataset files:", path)

Path to dataset files: /Users/gangaaram/.cache/kagglehub/datasets/raghavendragandhi/retail-customer-and-transaction-dataset/versions/1


In [4]:
import os

print(os.listdir(path))

['customers.csv', 'customer_reviews_complete.csv', 'interactions.csv', 'transactions.csv', 'campaigns.csv', 'support_tickets.csv']


In [61]:
import pandas as pd 

#Retrieving relevant datasets
customers = pd.read_csv(os.path.join(path, "customers.csv"))
transactions = pd.read_csv(os.path.join(path, "transactions.csv"))
interactions = pd.read_csv(os.path.join(path, "interactions.csv"))
support_tickets = pd.read_csv(os.path.join(path, "support_tickets.csv"))
campaigns = pd.read_csv(os.path.join(path, "campaigns.csv"))

# Clean and format data

In [9]:
# Changing to date time
transactions['transaction_date']=pd.to_datetime(transactions['transaction_date'])
interactions['interaction_date']=pd.to_datetime(interactions['interaction_date'])
support_tickets['submission_date']=pd.to_datetime(support_tickets['submission_date'])
support_tickets['resolution_date']=pd.to_datetime(support_tickets['resolution_date'])

In [11]:
transactions.isna().sum()

transaction_id        0
customer_id           0
product_name        678
product_category    686
quantity            644
price               622
transaction_date      0
store_location      644
payment_method      660
discount_applied    611
dtype: int64

In [13]:
#Dropping null values
transactions=transactions.dropna(subset=['product_name', 'product_category',
                                         'quantity', 'price', 'store_location',
                                         'payment_method'])

In [15]:
#Replace discount applied na with 0
transactions['discount_applied'].value_counts(dropna=False)
transactions['discount_applied']=transactions['discount_applied'].fillna(0)

In [17]:
interactions.isna().sum()

interaction_id         0
customer_id            0
channel             2002
interaction_type    2022
interaction_date       0
duration            1963
page_or_product     1927
session_id             0
dtype: int64

In [19]:
#Dropping null values
interactions=interactions.dropna(subset='interaction_type')

In [21]:
support_tickets.isna().sum()

ticket_id                        0
customer_id                      0
issue_category                  71
priority                        60
submission_date                  0
resolution_date                244
resolution_status               57
resolution_time_hours          299
customer_satisfaction_score    341
notes                           61
dtype: int64

In [23]:
#Dropping null values
support_tickets=support_tickets.dropna(subset='issue_category')

In [25]:
#Checking for duplicate primary keys and entries
customers['customer_id'].duplicated().sum()
transactions['transaction_id'].duplicated().sum()
interactions['interaction_id'].duplicated().sum()
transactions.duplicated().sum()
customers.duplicated().sum()
interactions.duplicated().sum()

0

In [27]:
#Remove customers with 0 transactions
customer_transactions = customers.merge(transactions, on='customer_id',how='left')

customer_transactions.insert(3,
                             'no_of_transactions',
                             customer_transactions.groupby('customer_id')['transaction_id']
                             .transform('count') #follows format of original rows
                            )

customer_transactions=customer_transactions[customer_transactions['no_of_transactions']!=0]

# Begin Analysis. Find out top 10% of customers with most number of transactions

In [30]:
customer_transactions.insert(18,
                     'total_trans_value',
                     customer_transactions['price']*customer_transactions['quantity']
                    )

desc_transactions_all=(customer_transactions.sort_values(by='no_of_transactions',ascending=False))                     
                     
unique_customers= (desc_transactions_all
    .drop_duplicates(subset=['customer_id']) #customer_id is primary key (unique)
)

top_10_transactions_people = unique_customers.head( int(len(unique_customers) * 0.10)
                    ).copy() #dataframe of top 10 customers

print(len(unique_customers)) # total number of unique customers
top_10_transactions_people.head()

4565


,customer_id,full_name,age,no_of_transactions,gender,email,phone,street_address,city,state,...,transaction_id,product_name,product_category,quantity,price,total_trans_value,transaction_date,store_location,payment_method,discount_applied
25395,1580c83f-3afc-4195-b4ae-923acac7e5ef,Larry Pena,39.0,29,Male,larry.pena@hotmail.com,+1-825-419-1756x431,303 Peter Run,Orlando,Florida,...,e33f9090-b8b9-4af6-93c0-6648e84f0f88,Bed Frame,Furniture,2.0,886.87,1773.74,2021-12-05,"Chicago, IL",Credit Card,15.0
23399,a1e0a52f-24ea-4ac2-952e-85ca9b770bea,Nicole Owen,41.0,25,Female,nicole.owen@yahoo.com,001-488-352-4967,79370 Gonzalez Streets,Los Angeles,California,...,0bf486e7-fa89-4096-b8be-0da814a5e7c0,Ring Doorbell,Smart Home Devices,1.0,283.62,283.62,2022-08-02,Online,Apple Pay,0.0
21365,715a33b1-a571-4cea-a1ce-162eb50a6361,Logan Scott,31.0,25,Male,scottl@yahoo.com,359-807-2667,9393 Tara Rue,Fort Worth,Texas,...,0441a714-61bd-4881-a892-deba49a18e78,Ring Doorbell,Smart Home Devices,1.0,139.59,139.59,2020-11-29,"San Francisco, CA",PayPal,20.0
15659,6f9cd25b-df62-4938-aa39-1db5262bf59c,Kelly Blevins,40.0,24,Female,kblevins@yahoo.com,(574)437-5982x481,17744 Rodriguez Villages,Pittsburgh,Pennsylvania,...,fccc36b9-e215-4ea7-90af-8cf1cad9ab0e,Wall Art,Home Decor,1.0,220.86,220.86,2021-08-20,"New York, NY",PayPal,0.0
27422,01f30800-c880-44d9-a114-4219284844b3,Brian Murray,35.0,24,Male,brian_murray@gmail.com,001-858-606-1150x38584,300 Ford Mountain,Tallahassee,Florida,...,2a11987f-f8d9-4623-8671-decc41f33765,Ring Doorbell,Smart Home Devices,1.0,115.22,115.22,2025-01-04,"Seattle, WA",PayPal,0.0


# Find out if customers who are most frequent also spend the most 

In [33]:
import numpy as np

agg_customers=(customer_transactions.groupby(['customer_id','full_name'])
               .agg(total_trans_value=('total_trans_value','sum'),
               no_of_transactions=('transaction_id','count')).reset_index() #make customer_id and full name into individual columns
              )
                                          
agg_customers=agg_customers.sort_values('no_of_transactions',ascending=False).reset_index(drop=True) #remove new index 

agg_customers['percentile']=np.arange(len(agg_customers))/len(agg_customers)  
# 0,1,2,3..5000/ 5000, gets percentile so first person is 0 percentile

bins=np.arange(0,1.1,0.1)  #[0,0.1,0.2...1.0] required 11 bin boundaries 0 included in first bin, 1 in last bin

labels=[f"{i}-{i+10}%" for i in range (0,100,10)]

agg_customers['transactions_percentile_group']=pd.cut(agg_customers['percentile'],
                                         bins=bins,
                                         labels=labels,
                                         right=True, #e.g. (0.9-1] 
                                         include_lowest=True) #e.g. [0-0.1] lowest bin left side is included

transaction_value_data=(agg_customers.groupby('transactions_percentile_group',observed=True)['total_trans_value']
       .mean()                                                                #observed here removes any percentile groups with 0 rows
       .reset_index()
                       )

transaction_value_data['percentage_of_total_revenue']=(transaction_value_data['total_trans_value']/
                                                       transaction_value_data['total_trans_value'].sum()
                                                      )

transaction_value_data

,transactions_percentile_group,total_trans_value,percentage_of_total_revenue
0,0-10%,13103.111923,0.233361
1,10-20%,9910.999250,0.176511
2,20-30%,7958.723327,0.141742
3,30-40%,6130.274126,0.109178
4,40-50%,5051.442723,0.089964
5,50-60%,4650.924566,0.082831
6,60-70%,4032.287490,0.071813
7,70-80%,2468.393579,0.043961
8,80-90%,1906.161134,0.033948
9,90-100%,937.229779,0.016692


#### Findings: So we find that people who are frequent customers also spend more. Therefore both no_of_transactions and total_trans can be used to analyse demographics and as proxies for each other.

# Finding out the demographic of top 10% of frequent customers and analysing purchasing patterns

In [39]:
#Find out gender ratios
countsgender=top_10_transactions_people['gender'].value_counts()

percentagegender=top_10_transactions_people['gender'].value_counts(normalize=True)*100

totaltransgender=top_10_transactions_people.groupby('gender')['total_trans_value'].sum()

trans_percentage_gender=((top_10_transactions_people.groupby('gender')['total_trans_value'].sum())/
                         top_10_transactions_people['total_trans_value'].sum()
                        )

gender_summary=(pd.concat([countsgender,percentagegender,totaltransgender,trans_percentage_gender]
                          ,axis=1,keys=['no_of_transactions','no_of_transactions_percentage',
                                        'total_trans_value','total_trans_value_percentage'])
               )

gender_summary
#Not much difference in gender spending

,no_of_transactions,no_of_transactions_percentage,total_trans_value,total_trans_value_percentage
gender,,,,
Female,218,49.099099,167595.911867,0.445961
Male,206,46.396396,178785.850000,0.475737
Non-binary,15,3.378378,11639.340000,0.030971
Prefer not to say,5,1.126126,1671.960000,0.004449


In [41]:
#Split by states
countsstate=top_10_transactions_people['state'].value_counts()

percentagestate=top_10_transactions_people['state'].value_counts(normalize=True)*100

totaltransstate=top_10_transactions_people.groupby('state')['total_trans_value'].sum()

trans_percentage_state=((top_10_transactions_people.groupby('state')['total_trans_value'].sum())/
                        top_10_transactions_people['total_trans_value'].sum()
                       )
state_summary=(pd.concat([countsstate,percentagestate,totaltransstate,trans_percentage_state],axis=1,
                         keys=['no_of_transactions','no_of_transactions_percentage',
                               'total_trans_value','total_trans_value_percentage'])
              )
state_summary

#Frequent customers come from California

,no_of_transactions,no_of_transactions_percentage,total_trans_value,total_trans_value_percentage
state,,,,
California,86,19.111111,77232.589646,0.205511
Texas,56,12.444444,39440.010000,0.104947
New York,54,12.000000,35133.110000,0.093487
Florida,53,11.777778,49000.220000,0.130386
Ohio,29,6.444444,24922.944133,0.066318
Pennsylvania,27,6.000000,21359.822410,0.056837
Michigan,23,5.111111,13381.270000,0.035607
North Carolina,21,4.666667,19000.600000,0.050559
Illinois,20,4.444444,28991.635677,0.077145


In [158]:
#Find out preferred channel
countschannel=top_10_transactions_people['preferred_channel'].value_counts()
percentagechannel=top_10_transactions_people['preferred_channel'].value_counts(normalize=True)*100
channel_summary=pd.concat([countschannel,percentagechannel],axis=1,keys=['no_of_transactions','percentage'])
channel_summary

#Not much difference in channels

,no_of_transactions,percentage
preferred_channel,,
both,221,48.893805
online,198,43.805310
in-store,33,7.300885


In [43]:
#Split by age groups
bins =range(0,101,10)
labels=[f'{i}-{i+9}' for i in range (0,91,10)]

top_10_transactions_people['age_group']=(pd.cut(top_10_transactions_people['age']
                                 ,labels=labels
                                 ,bins=bins
                                 ,right=False
                                )
                         )
age_summary=(top_10_transactions_people
             .groupby('age_group',observed=False)
             .agg(no_of_transactions=('no_of_transactions','sum'),
                total_trans_value=('total_trans_value','sum')
                 )
             .reset_index()
            )
                                                                                  
                                                               
age_summary['nooftrans_percentage']=(age_summary['no_of_transactions']/age_summary['no_of_transactions'].sum()) * 100
age_summary['transval_percentage']=(age_summary['total_trans_value']/age_summary['total_trans_value'].sum()) * 100
                                                                
age_summary
#Age groups 20-29,30-39,40-49 are most frequent and high value spenders

,age_group,no_of_transactions,total_trans_value,nooftrans_percentage,transval_percentage
0,0-9,0,0.000000,0.000000,0.000000
1,10-19,38,2599.200000,0.588600,0.757853
2,20-29,1453,76406.830746,22.506196,22.278066
3,30-39,3055,164706.039081,47.320322,48.023612
4,40-49,1711,86489.506363,26.502478,25.217889
5,50-59,180,11907.670000,2.788104,3.471939
6,60-69,19,859.620000,0.294300,0.250641
7,70-79,0,0.000000,0.000000,0.000000
8,80-89,0,0.000000,0.000000,0.000000
9,90-99,0,0.000000,0.000000,0.000000


### Investigate ages 20-49 for marketing campaigns

In [117]:
nonnullage_top10_transactions_people=top_10_transactions_people[top_10_transactions_people['age'].notnull()]
twostofours=nonnullage_top10_transactions_people[nonnullage_top10_transactions_people['age'].between(20,49,inclusive='both')]

In [145]:
#Find out customer segmentations
campaigns['target_segment'].value_counts()

target_segment
East Coast                16
Kitchen Enthusiasts       15
Southern States           14
Middle-aged (41-60)       14
In-Store Shoppers         13
Seniors (60+)             12
Adults (26-40)            12
Technology Enthusiasts    12
Home Improvement          11
High-Value Customers      10
All Customers             10
Loyal Customers           10
New Customers              9
Young Adults (18-25)       9
Inactive Customers         9
West Coast                 8
Online Shoppers            8
Midwest                    8
Name: count, dtype: int64

In [147]:
twostofours['state'].value_counts()
#States are already being prioritised with marketing campaigns, so explore other data.

state
California        79
Texas             53
Florida           47
New York          47
Ohio              27
Pennsylvania      25
Michigan          22
North Carolina    20
Illinois          18
New Jersey        15
Georgia           15
Virginia          13
Massachusetts     13
Arizona            9
Washington         9
Name: count, dtype: int64

In [149]:
#We notice that high-value customers, loyal customers and adults can be improved.
twostofours['preferred_channel'].value_counts()

preferred_channel
both        198
online      185
in-store     30
Name: count, dtype: int64

In [151]:
campaigns[campaigns['target_segment']=='Adults (26-40)']['campaign_type'].value_counts()

campaign_type
Radio Advertisement     3
Print Advertisement     2
Email Marketing         2
SMS Marketing           1
Influencer Marketing    1
Online Display Ads      1
TV Advertisement        1
Name: count, dtype: int64

In [139]:
campaigns[campaigns['target_segment']=='High-Value Customers']['campaign_type'].value_counts()

campaign_type
Radio Advertisement     2
Influencer Marketing    2
Online Display Ads      2
Print Advertisement     1
In-Store Promotion      1
Social Media            1
SMS Marketing           1
Name: count, dtype: int64

In [137]:
campaigns[campaigns['target_segment']=='Loyal Customers']['campaign_type'].value_counts()

campaign_type
Print Advertisement    4
Online Display Ads     3
SMS Marketing          1
Radio Advertisement    1
Social Media           1
Name: count, dtype: int64

### Findings: The 3 customers segments : Adults, High-value Customers and Loyal customers have fewer marketing campaigns. The campaign types do not match their preferred channels. Online and instore marketing campaign types rank the lowest for each customer segment.

# Find out what are the products and categories which have the most sales for top spenders and one-time customers

In [49]:
top_10_transactions_people_items=(customer_transactions.sort_values(by='no_of_transactions',ascending=False)
                    )
#Top categories
countscategories=top_10_transactions_people_items['product_category'].value_counts()

percentagecategories=top_10_transactions_people_items['product_category'].value_counts(normalize=True)*100

categories_summary=pd.concat([countscategories,percentagecategories],axis=1,keys=['no_of_transactions','percentage'])

categories_summary

,no_of_transactions,percentage
product_category,,
Smartphones,3959,13.866414
Smart Home Devices,3905,13.677279
Furniture,3511,12.297293
Home Decor,3351,11.736892
Kitchen Appliances,3328,11.656334
Laptops,1725,6.041820
Audio Equipment,1705,5.971770
Gaming Consoles,1698,5.947252
TVs,1228,4.301075


In [51]:
#Finding top items in each category

countsitems=(top_10_transactions_people_items.groupby('product_category')['product_name']
           .value_counts()
           .groupby(level=0) # this regroups back into categories so that we get top 3 in each category instead of just first 3
           .head(3)
          )

percentageitems=((top_10_transactions_people_items.groupby('product_category')['product_name']
           .value_counts(normalize=True)*100)
           .groupby(level=0) 
           .head(3)
          )

top_items=pd.concat([countsitems,percentageitems],axis=1,keys=['no_of_transactions','percentage'])
top_items.reindex(countscategories.index,level=0) #.index gives us the labels of above, 
#reindex to follow the biggest to lowest categ and level=0


no_of_transactions  \
product_category         product_name                                   
Smartphones              iPhone 13                                827   
                         OnePlus 10                               810   
                         Xiaomi Mi 12                             809   
Smart Home Devices       Amazon Echo                              807   
                         Google Nest                              806   
                         Smart Thermostat                         790   
Furniture                Bed Frame                                743   
                         Office Desk                              701   
                         Dining Table                             697   
Home Decor               Throw Pillows                            707   
                         Area Rug                                 697   
                         Table Lamp                               665   
Kitchen Appliances       Electric Range                           686   
                         Microwave Oven                           686   
                         Range Hood                               674   
Laptops                  Lenovo ThinkPad                          365   
                         HP Spectre                               353   
                         Asus ZenBook                             343   
Audio Equipment          Audio-Technica Turntable                 366   
                         Sony Soundbar                            345   
                         Sonos Speaker                            344   
Gaming Consoles          Oculus Quest                             357   
                         Nintendo Switch                          337   
                         PlayStation 5                            336   
TVs                      Vizio SmartCast TV                       267   
                         LG OLED TV                               255   
                         Sony Bravia                              254   
Small Kitchen Appliances Food Processor                           265   
                         Toaster                                  249   
                         Coffee Maker                             248   
Desktop Computers        Asus ROG                                 130   
                         Dell Inspiron Desktop                    129   
                         HP Pavilion                              125   
Tablets                  Amazon Fire HD                           126   
                         Microsoft Surface                        121   
                         iPad Pro                                 120   
Cookware                 Baking Sheet                             129   
                         Dutch Oven                               126   
                         Cookware Set                             118   
Bedding                  Duvet Cover                              134   
                         Pillows                                  126   
                         Sheets                                   109   
Computer Accessories     External Hard Drive                      116   
                         USB-C Hub                                115   
                         Logitech Mouse                           112   

                                                   percentage  
product_category         product_name                          
Smartphones              iPhone 13                  20.889113  
                         OnePlus 10                 20.459712  
                         Xiaomi Mi 12               20.434453  
Smart Home Devices       Amazon Echo                20.665813  
                         Google Nest                20.640205  
                         Smart Thermostat           20.230474  
Furniture                Bed Frame                  21.162062  
                         Office Desk       

### Finding one - time customers

In [54]:
one_time_customers = desc_transactions_all[
    desc_transactions_all['no_of_transactions'] == 1
]

one_time_customers_ids=one_time_customers['customer_id']

one_time_customers.head()

,customer_id,full_name,age,no_of_transactions,gender,email,phone,street_address,city,state,...,transaction_id,product_name,product_category,quantity,price,total_trans_value,transaction_date,store_location,payment_method,discount_applied
25532,7c3570c1-7836-48a1-9ed4-bbd211e89682,Michael Moreno,21.0,1,Male,mmoreno@hotmail.com,001-861-978-8269x602,1453 Burke Lake Suite 042,Tallahassee,Florida,...,758f80b2-2f1f-4017-b6a3-94ae89899757,Smart Thermostat,Smart Home Devices,1.0,55.19,55.19,2025-02-01,Online,PayPal,5.0
17896,59a643c0-a2a4-4805-8850-e37151aa00bf,Kimberly Anderson,45.0,1,Female,kimberly.anderson@hotmail.com,+1-836-414-3269x9537,922 Mary Lodge Apt. 375,Albany,New York,...,f1f7a132-8e43-4d5a-93e3-908067cb0a11,Table Lamp,Home Decor,1.0,240.04,240.04,2024-11-13,Online,Debit Card,0.0
13792,b4b70f50-08c3-431b-80fd-b457c4ff65f1,Jeff Hamilton,28.0,1,Male,jhamilton@yahoo.com,733.985.6836,3799 Perez Corners,Phoenix,Arizona,...,62932364-58e4-4ff5-b235-533ae330d120,Xbox Series X,Gaming Consoles,1.0,272.50,272.50,2023-02-22,Online,Debit Card,0.0
21235,9c969873-d1cd-4468-9f05-f30a41a9dc1a,Jeffrey Bennett,49.0,1,Male,jbennett@gmail.com,001-452-489-2167,9992 Howard Radial,Orlando,Florida,...,d0d5aa8c-499f-49a6-a505-a73858a8b8fb,Curtains,Home Decor,1.0,69.06,69.06,2024-09-29,Online,Debit Card,10.0
1460,edc78c24-0f12-407b-a6d9-97aca3b83f6e,Elizabeth Foster,40.0,1,Female,fostere@hotmail.com,662.741.5027x45644,NaN,Vancouver,Washington,...,0dd99ff9-8a15-40c3-987f-49f370363d73,Throw Pillows,Home Decor,1.0,30.96,30.96,2024-12-09,Online,PayPal,5.0


### Finding differences between one-time customers and frequent customers

In [57]:
#Top categories
countscategoriesone=one_time_customers['product_category'].value_counts()
percentagecategoriesone=one_time_customers['product_category'].value_counts(normalize=True)*100
categories_summary=pd.concat([countscategoriesone,percentagecategoriesone],axis=1,keys=['no_of_transactions','percentage'])
categories_summary

#Similar product category order between one-time customers and repeat customers.

,no_of_transactions,percentage
product_category,,
Smart Home Devices,68,14.879650
Smartphones,55,12.035011
Home Decor,47,10.284464
Kitchen Appliances,46,10.065646
Furniture,43,9.409190
Gaming Consoles,35,7.658643
Laptops,35,7.658643
Audio Equipment,28,6.126915
Small Kitchen Appliances,23,5.032823


In [59]:
countsitems=(one_time_customers.groupby('product_category')['product_name']
           .value_counts()
           .groupby(level=0) 
           .head(3)
          )
percentageitems=((one_time_customers.groupby('product_category')['product_name']
           .value_counts(normalize=True)*100)
           .groupby(level=0) 
           .head(3)
          )
top_items=pd.concat([countsitems,percentageitems],axis=1,keys=['no_of_transactions','percentage'])
top_items.reindex(countscategories.index,level=0) 

#Similar top products as well.

no_of_transactions  \
product_category         product_name                                   
Smartphones              iPhone 13                                 15   
                         Xiaomi Mi 12                              13   
                         Google Pixel 6                            10   
Smart Home Devices       Smart Thermostat                          17   
                         Ring Doorbell                             16   
                         Philips Hue Lights                        15   
Furniture                Office Desk                               16   
                         Dining Table                               9   
                         Bookshelf                                  7   
Home Decor               Area Rug                                  14   
                         Curtains                                  10   
                         Throw Pillows                              9   
Kitchen Appliances       Microwave Oven                            16   
                         Electric Range                            12   
                         Refrigerator                               8   
Laptops                  Lenovo ThinkPad                            9   
                         Asus ZenBook                               8   
                         HP Spectre                                 8   
Audio Equipment          Bose Headphones                            9   
                         JBL Bluetooth Speaker                      6   
                         Audio-Technica Turntable                   5   
Gaming Consoles          PlayStation 5                             10   
                         Oculus Quest                               7   
                         Xbox Series X                              7   
TVs                      Samsung QLED TV                            8   
                         Vizio SmartCast TV                         7   
                         Sony Bravia                                6   
Small Kitchen Appliances Toaster                                    8   
                         Air Fryer                                  5   
                         Coffee Maker                               5   
Desktop Computers        Asus ROG                                   7   
                         Dell Inspiron Desktop                      3   
                         HP Pavilion                                3   
Tablets                  Lenovo Tab                                 2   
                         Microsoft Surface                          2   
                         Samsung Galaxy Tab                         2   
Cookware                 Baking Sheet                               4   
                         Knife Set                                  3   
                         Cookware Set                               2   
Bedding                  Mattress Topper                            4   
                         Pillows                                    3   
                         Sheets                                     3   
Computer Accessories     External Hard Drive                        4   
                         USB-C Hub                                  3   
                         Logitech Mouse                             2   

                                                   percentage  
product_category         product_name                          
Smartphones              iPhone 13                  27.272727  
                         Xiaomi Mi 12               23.636364  
                         Google Pixel 6             18.181818  
Smart Home Devices       Smart Thermostat           25.000000  
                         Ring Doorbell              23.529412  
                         Philips Hue Lights         22.058824  
Furniture                Office Desk                37.209302  
                         Dining Table      

### Findings: There are not much differences between products and categories.

# Investigate customer interactions

In [163]:
#Finding percentage of interactions for each group
customer_interactions=customers.merge(interactions,on='customer_id',how='left')

repeat_customers= (desc_transactions_all[desc_transactions_all['no_of_transactions']>1]
    .drop_duplicates(subset=['customer_id'])
)

one_time_customer_interactions = customer_interactions[
    customer_interactions['customer_id'].isin(one_time_customers_ids)
]

repeat_ids=repeat_customers['customer_id']

repeat_customers_interactions=customer_interactions[customer_interactions['customer_id'].isin(repeat_ids)]

repeat_interaction_percentage=((repeat_customers_interactions['interaction_type']
                                .value_counts(normalize=True)*100).reset_index(name='repeat')
                              )

one_time_customer_interactions=customer_interactions[customer_interactions['customer_id'].isin(one_time_customers_ids)]

onetime_interaction_percentage=((one_time_customer_interactions['interaction_type']
                                 .value_counts(normalize=True)*100).reset_index(name='one_time')
                               )

interaction_percentage_by_group=repeat_interaction_percentage.merge(onetime_interaction_percentage,on='interaction_type',how='left')

interaction_percentage_by_group

#We find that checkout and product view are the highest interactions for both repeat and one_time and interaction ratios are similar.

,interaction_type,repeat,one_time
0,checkout,13.438322,13.794999
1,product_view,12.928392,12.826326
2,search,10.609737,10.806540
3,wishlist_add,10.590850,10.875240
4,add_to_cart,10.509494,10.353119
5,purchase,10.509494,10.414949
6,review,7.006814,7.749382
7,page_view,6.871704,8.099753
8,app_open,3.507039,3.187689
9,notification_click,3.472172,3.510580


In [165]:
#We now find the average number of interactions per customer in both groups.

repeat_customers_avg_interactions = (
    repeat_customers_interactions
    .groupby('interaction_type')['customer_id']
    .count()
    / len(repeat_ids)
) # Finding total number of interactions then dividing by number of customers in group

repeat_customers_avg_interactions = repeat_customers_avg_interactions.reset_index(name='avg_per_customer')

onetime_customers_avg_interactions = (
    one_time_customer_interactions
    .groupby('interaction_type')['customer_id']
    .count()
    / len(one_time_customers_ids)
)

onetime_customers_avg_interactions = onetime_customers_avg_interactions.reset_index(name='avg_per_customer')


customers_avg_interactions = repeat_customers_avg_interactions.merge(
    onetime_customers_avg_interactions,
    on='interaction_type',
    how='outer',
    suffixes=('_repeat', '_one_time')
).fillna(0)

customers_avg_interactions 

#We find that the average onetime user has higher interactions, notably a near 2x difference in checkout & purchase interactions.

,interaction_type,avg_per_customer_repeat,avg_per_customer_one_time
0,add_to_cart,1.760954,3.297593
1,app_open,0.587634,1.015317
2,checkout,2.251704,4.393873
3,inventory_check,0.479309,0.750547
4,notification_click,0.581792,1.118162
5,page_view,1.151412,2.579869
6,product_lookup,0.416261,0.706783
7,product_view,2.166261,4.085339
8,purchase,1.760954,3.317287
9,review,1.174051,2.468271


# Find out if purchase interactions mean completed transactions

### As the one time users only purchase once, logically it doesn't make sense for them to have more purchase interactions, therefore purchases interactions may not equate to completed transactions


In [167]:
#We investigate one person from the onetime purchase group

customer_id = one_time_customers.iloc[2]['customer_id']

interactions[
   ( interactions['customer_id'] == customer_id )& (interactions['interaction_type']=='purchase')
].sort_values('interaction_date').head()

,interaction_id,customer_id,channel,interaction_type,interaction_date,duration,page_or_product,session_id
68726,7c6db303-c6ef-4999-9c0e-78bf48f8cfb5,b4b70f50-08c3-431b-80fd-b457c4ff65f1,web,purchase,2023-03-02 05:36:00,214.0,category_furniture,b4b70f50-08c3-431b-80fd-b457c4ff65f1_session_1
68728,590e83e7-9d79-41fc-a977-efaa06b83cab,b4b70f50-08c3-431b-80fd-b457c4ff65f1,web,purchase,2023-04-05 10:14:00,170.0,blog,b4b70f50-08c3-431b-80fd-b457c4ff65f1_session_3
68732,77e759bf-1f2e-46a1-861d-e2f00b3d110e,b4b70f50-08c3-431b-80fd-b457c4ff65f1,web,purchase,2023-04-16 13:15:00,193.0,blog,b4b70f50-08c3-431b-80fd-b457c4ff65f1_session_7
68733,c55e33b2-3bf4-44e4-a3ea-9cd9b23db309,b4b70f50-08c3-431b-80fd-b457c4ff65f1,web,purchase,2023-06-06 15:01:00,161.0,category_laptops,b4b70f50-08c3-431b-80fd-b457c4ff65f1_session_8
68750,d0de608e-4c1e-4bb7-a2d3-7aae198f17af,b4b70f50-08c3-431b-80fd-b457c4ff65f1,web,purchase,2024-01-22 05:29:00,217.0,about_us,b4b70f50-08c3-431b-80fd-b457c4ff65f1_session_25


In [169]:
transactions[
    transactions['customer_id'] == customer_id
].sort_values('transaction_date')
#We find that the number of actual transactions (below) are lesser than purchase interactions(above) so they are not the same.

,transaction_id,customer_id,product_name,product_category,quantity,price,transaction_date,store_location,payment_method,discount_applied
12532,62932364-58e4-4ff5-b235-533ae330d120,b4b70f50-08c3-431b-80fd-b457c4ff65f1,Xbox Series X,Gaming Consoles,1.0,272.5,2023-02-22,Online,Debit Card,0.0


In [173]:
#We analyse the whole one_time_customer group

# Total purchase interactions of one time customers
one_time_purchase_interactions = (
    one_time_customer_interactions[one_time_customer_interactions['interaction_type'] == 'purchase']
    .groupby('customer_id')
    .size() #includes totl row count including null values
    .rename('purchase_interactions')
)

# Total transactions by one time customers

one_time_transactions = transactions[transactions['customer_id'].isin(one_time_customers_ids)]

one_time_actual_transactions = (
    one_time_transactions
    .groupby('customer_id')
    .size()
    .rename('actual_transactions')
)

# Compare them
one_time_comparison = pd.concat(
    [one_time_purchase_interactions, one_time_actual_transactions],
    axis=1
).fillna(0)

one_time_comparison.head()

,purchase_interactions,actual_transactions
customer_id,,
01c13f1d-9b42-44cf-a1fc-c6c63110a1c2,2.0,1
02152e28-069f-4aaf-a95d-404c23522666,3.0,1
03080105-fbe7-44eb-8366-a0207a90e47a,6.0,1
044631e7-fe03-437b-a452-8e81742b6177,2.0,1
0614f4df-997d-4fad-a96b-bb3442fccab0,6.0,2


### So we find that one time purchase interactions arent the same as actual transactions so this means people are exiting from the final stage of payment when keying in card details. We now find out how significant this problem is.

In [183]:
#We find the number of one time customers who have more purchase interactions than actual transactions
one_time_comparison['difference'] = (
    one_time_comparison['purchase_interactions']
    - one_time_comparison['actual_transactions']
)

one_time_comparison = one_time_comparison.sort_values(
    'difference',
    ascending=False
)

(one_time_comparison['purchase_interactions'] >
 one_time_comparison['actual_transactions']).mean() * 100

#66% of one time customers have more purchase interactions then actual transactions.

65.86433260393873

In [185]:
#We analyse the repeat customer group.

# Total purchase interactions of repeat customers
repeat_purchase_interactions = (
    repeat_customers_interactions[repeat_customers_interactions['interaction_type'] == 'purchase']
    .groupby('customer_id')
    .size() #includes totl row count including null values
    .rename('purchase_interactions')
)

# Total transactions by one time customers

repeat_transactions = transactions[transactions['customer_id'].isin(repeat_ids)]

repeat_actual_transactions = (
    repeat_transactions
    .groupby('customer_id')
    .size()
    .rename('actual_transactions')
)

# Compare them
repeat_comparison = pd.concat(
    [repeat_purchase_interactions, repeat_actual_transactions],
    axis=1
).fillna(0)

repeat_comparison['difference'] = (
    repeat_comparison['purchase_interactions']
    - repeat_comparison['actual_transactions']
)

repeat_comparison = repeat_comparison.sort_values(
    'difference',
    ascending=False
)

(repeat_comparison['purchase_interactions'] >
repeat_comparison['actual_transactions']).mean() * 100

#Only 9% of repeat customers have more purchase interactions then actual transactions.

9.152872444011685

### Findings: Repeat customers are significantly (57%) more likely to complete their purchase interactions. Therefore we have to find solutions to encourage one time customers to complete their transaction.

# Comparing customer interactions after their first transcations

### Finding purchasing frequency between repeat customers

In [188]:
#Sort by customer id and then transaction_date
repeat_transactions=repeat_transactions.sort_values(['customer_id','transaction_date'])

#Convert to pd.date
repeat_transactions['transaction_date']=pd.to_datetime(repeat_transactions['transaction_date'])

#Find difference in date
repeat_transactions['diff_in_date']=repeat_transactions.groupby('customer_id')['transaction_date'].diff().dt.days

#Find average
repeat_transactions['avg_diff_in_date']=repeat_transactions['diff_in_date'].mean()

# Calculate all metrics at once
diff_stats = repeat_transactions['diff_in_date'].agg(
    mean='mean',
    median='median',
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75),
    p90=lambda x: x.quantile(0.90)
)

print(diff_stats)

#So we find that the mean number of days between transactions is 117 days for repeat customers. 

mean      117.473071
median     79.000000
p25        31.000000
p75       161.000000
p90       274.000000
Name: diff_in_date, dtype: float64


### Find all transaction dates for the repeat and one time customers.

In [193]:
one_time_dates=one_time_customers[['customer_id','transaction_date']]
repeat_dates=repeat_transactions[['customer_id','transaction_date']]

### Find number of high priority interactions after the purchase dates.

In [197]:
one_time_interactions_dates = one_time_customer_interactions.merge(
    one_time_dates,
    on='customer_id',
    how='left'
)

# Convert dates to datetime
one_time_interactions_dates['interaction_date'] = pd.to_datetime(
    one_time_interactions_dates['interaction_date']
)

one_time_interactions_dates['transaction_date'] = pd.to_datetime(
    one_time_interactions_dates['transaction_date']
)

# Keep only interactions after the customer's transaction
one_time_interactions_dates = (
    one_time_interactions_dates[
        one_time_interactions_dates['interaction_date'] >
        one_time_interactions_dates['transaction_date']
    ]
    .sort_values(['customer_id', 'interaction_date'])
    .copy()
)

#Find number of days after transaction
one_time_interactions_dates['diff_in_dates'] = (
    one_time_interactions_dates['interaction_date'] -
    one_time_interactions_dates['transaction_date']
).dt.days

#Define high priority interactions
high_priority = [
    'product_view',
    'wishlist_add',
    'add_to_cart',
    'checkout',
    'purchase'
]

# 4. Calculate % of customers performing each interaction within 30/60/90/120<-(mean days between transactions for repeat customers) days
windows = [30, 60, 90, 120]

summaries = []

for days in windows:

    # Keep interactions within the specified window
    df = one_time_interactions_dates[
        one_time_interactions_dates['diff_in_dates'] <= days
    ]

    # Keep only high-priority interactions
    high = df[
        df['interaction_type'].isin(high_priority)
    ]

    # Count unique customers performing each interaction
    summary = (
        high
        .groupby('interaction_type')['customer_id']
        .nunique()
        .div(len(one_time_customers_ids))
        .mul(100)
        .reset_index(name=f'Within {days} days')
    )

    summaries.append(summary)

#Combine all results for 30/60/90/120 daus
one_time_summary = summaries[0]

for summary in summaries[1:]:
    one_time_summary = one_time_summary.merge(
        summary,
        on='interaction_type',
        how='outer'
    )

# Ensures that its in the right order
one_time_summary['interaction_type'] = pd.Categorical(
    one_time_summary['interaction_type'],
    categories=high_priority, #set to follow
    ordered=True # follows the index so not alphabetical sort
)

one_time_summary = (
    one_time_summary
    .sort_values('interaction_type')
    .reset_index(drop=True)
    .rename(columns={'interaction_type':'interaction_type_onetime'})
)

one_time_summary

,interaction_type_onetime,Within 30 days,Within 60 days,Within 90 days,Within 120 days
0,product_view,36.323851,51.859956,57.768053,60.175055
1,wishlist_add,36.761488,46.608315,50.765864,53.829322
2,add_to_cart,30.853392,43.763676,51.203501,53.610503
3,checkout,44.857768,56.455142,64.332604,67.833698
4,purchase,33.479212,46.170678,51.422319,54.048140


In [199]:
# Now we repeat the same for repeat customers

repeat_interactions_dates = repeat_customers_interactions.merge(
    repeat_dates,
    on='customer_id',
    how='left'
)


repeat_interactions_dates['interaction_date'] = pd.to_datetime(
   repeat_interactions_dates['interaction_date']
)

repeat_interactions_dates['transaction_date'] = pd.to_datetime(
    repeat_interactions_dates['transaction_date']
)

repeat_interactions_dates['diff_in_dates'] = (
   repeat_interactions_dates['interaction_date'] -
   repeat_interactions_dates['transaction_date']
).dt.days

repeat_interactions_dates = repeat_interactions_dates[
    repeat_interactions_dates['diff_in_dates'] > 0
].copy()

repeat_interactions_dates = repeat_interactions_dates.sort_values(
    ['customer_id', 'transaction_date', 'interaction_date']
)

# Creating summaries
windows = [30, 60, 90,120]
summaries = []

for days in windows:

    # IMPORTANT: always start from the original dataframe
    df = repeat_interactions_dates[
        repeat_interactions_dates['diff_in_dates'] <= days
    ]

    # Keep high-priority interactions
    high = df[
        df['interaction_type'].isin(high_priority)
    ]

    # Percentage of customers performing each interaction
    summary = (
        high
        .groupby('interaction_type')['customer_id']
        .nunique()
        .div(len(repeat_ids))
        .mul(100)
        .reset_index(
            name=f'Within {days} days'
        )
    )

    summaries.append(summary)


# Combine summaries
repeat_summary = summaries[0]

for summary in summaries[1:]:
    repeat_summary = repeat_summary.merge(
        summary,
        on='interaction_type',
        how='outer'
    )


# Order interactions
repeat_summary['interaction_type'] = pd.Categorical(
    repeat_summary['interaction_type'],
    categories=high_priority,
    ordered=True
)

repeat_summary = (
    repeat_summary
    .sort_values('interaction_type')
    .reset_index(drop=True)
    .rename(columns={'interaction_type':'interaction_type_repeat'})
)

pd.concat([repeat_summary,one_time_summary],axis=1)

,interaction_type_repeat,Within 30 days,Within 60 days,Within 90 days,Within 120 days,interaction_type_onetime,Within 30 days,Within 60 days,Within 90 days,Within 120 days
0,product_view,33.227848,47.297955,55.331061,60.759494,product_view,36.323851,51.859956,57.768053,60.175055
1,wishlist_add,28.042843,41.187926,48.515093,52.848101,wishlist_add,36.761488,46.608315,50.765864,53.829322
2,add_to_cart,26.509250,39.751704,47.760467,52.580331,add_to_cart,30.853392,43.763676,51.203501,53.610503
3,checkout,34.298929,48.904576,56.888997,62.098345,checkout,44.857768,56.455142,64.332604,67.833698
4,purchase,26.801363,40.993184,48.880234,53.481013,purchase,33.479212,46.170678,51.422319,54.048140


In [201]:
# Investigate checkout as it has the highest percentage difference
checkout_counts= (interactions[interactions['interaction_type']=='checkout']
                  .groupby('customer_id')
                  .size()
                  .rename('checkout_count')
                  .reset_index()
                 )
checkout_counts.head()

,customer_id,checkout_count
0,00012aa8-e99c-4e30-b3f6-1f7e36adc517,2
1,000feeed-f931-4908-b539-29ab57e595be,6
2,00107f62-0530-48ec-a56a-49d90944eafe,3
3,005ae094-2f68-4a6d-91f0-73b38b890692,1
4,00a6835b-c50a-49a9-a1b5-19d74d3e1863,2


In [203]:
checkout_counts['customer_group'] = np.where(
    checkout_counts['customer_id'].isin(one_time_customers_ids),
    'One-time', #True condition
    'Repeat' # False condition
)
checkout_counts.groupby('customer_group')['checkout_count'].agg(
    ['count', 'mean', 'median', 'max']
)

#One-time customers reach checkout much more frequently

,count,mean,median,max
customer_group,,,,
One-time,440,4.563636,4.0,14
Repeat,3776,2.967691,2.0,13


### Findings: One-time customers have much higher, high intent interactions specifically checkouts that increase with time after their first transaction. Therefore a solution has to be created to convert these interactions into repeat purchases.

# Lastly we look at the supports table to find the issues one time customers face.

In [206]:
# Get customers who have submitted at least one support ticket
support_customers = support_tickets['customer_id'].unique()

# Create customer-level support indicator
support_summary = pd.DataFrame({
    'customer_id': pd.concat([
        one_time_customers_ids,
        repeat_ids
    ]).unique()
})

support_summary['customer_group'] = np.where(
    support_summary['customer_id'].isin(one_time_customers_ids),
    'One-time',
    'Repeat'
)

support_summary['has_support_ticket'] = (
    support_summary['customer_id'].isin(support_customers)
)

# Calculate percentage in each group
support_incidence = (
    support_summary
    .groupby('customer_group')['has_support_ticket']
    .value_counts(normalize=True)
    .mul(100)
)

support_incidence

#We find that one time and repeat customers have similar proportions of customers who have submitted a support ticket.

customer_group  has_support_ticket
One-time        False                 52.078775
                True                  47.921225
Repeat          False                 56.742941
                True                  43.257059
Name: proportion, dtype: float64

In [208]:
support_with_group = support_tickets.copy()

support_with_group['customer_group'] = np.where(
    support_with_group['customer_id'].isin(one_time_customers_ids),
    'One-time',
    np.where(
        support_with_group['customer_id'].isin(repeat_ids),
        'Repeat',
        'Other'
    )
)

support_with_group = support_with_group[
    support_with_group['customer_group'].isin(['One-time', 'Repeat'])
]

issue_summary = (
    support_with_group
    .groupby(['issue_category', 'customer_group'])['customer_id']
    .nunique()
    .reset_index(name='customers')
)

issue_summary

#We find each issue type and find that one time customers face the most issue with billing.

,issue_category,customer_group,customers
0,Account Issue,One-time,39
1,Account Issue,Repeat,341
2,Billing,One-time,47
3,Billing,Repeat,320
4,Product Inquiry,One-time,39
5,Product Inquiry,Repeat,298
6,Returns,One-time,39
7,Returns,Repeat,326
8,Shipping,One-time,44
9,Shipping,Repeat,332


In [222]:
segment_sizes = {
    'One-time': len(one_time_customers_ids),
    'Repeat': len(repeat_ids)
}

issue_summary['segment_customers'] = (
    issue_summary['customer_group'].map(segment_sizes)
)

issue_summary['issue_rate'] = (
    issue_summary['customers']
    / issue_summary['segment_customers']
    * 100
)

issue_comparison = (
    issue_summary
    .pivot(
        index='issue_category',
        columns='customer_group',
        values='issue_rate'
    )
    .reset_index()
)

issue_comparison['difference'] = (
    issue_comparison['One-time']
    - issue_comparison['Repeat']
)

issue_comparison.sort_values(
    'difference',
    ascending=False
)

#The biggest difference is in billing.

customer_group,issue_category,One-time,Repeat,difference
1,Billing,10.284464,7.789679,2.494785
4,Shipping,9.628009,8.081792,1.546217
2,Product Inquiry,8.533917,7.254138,1.279779
3,Returns,8.533917,7.935735,0.598182
6,Website Issue,8.315098,7.911392,0.403706
0,Account Issue,8.533917,8.300876,0.233041
5,Technical,7.877462,7.984421,-0.106959


### Findings: One time customers' biggest issue is billing and when compared to repeat customers they have proportionately more. A solution has to be created to address billing issues 

# Key Findings from Analysis

### Age groups 20-29,30-39,40-49 are most frequent and high value spenders.

### The 3 customers segments : Adults, High-value Customers and Loyal customers have fewer marketing campaigns. The campaign types do not match their preferred channels. Online and instore marketing campaign types rank the lowest for each customer segment.

### Repeat customers are significantly (57%) more likely to complete their purchase interactions. Therefore we have to find solutions to encourage one time customers to complete their transaction.

### One-time customers have much higher, high intent interactions specifically checkouts that increase with time after their first transaction. Therefore a solution has to be created to convert these interactions into repeat purchases.

### One time customers' biggest issue is billing and when compared to repeat customers they have proportionately more. A solution has to be created to address billing issues¶